# 9.4

c)

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(seed=42)

In [5]:
df = pd.read_csv("../../Datasets_from_the_book/salmon.dat", sep="\s+", dtype=np.float64)
categories = df.columns.tolist()
data = df.to_numpy()

In [ ]:
# The estimator for the stabilization point R = S

def theta_hat(beta1, beta2):
    return (1 - beta2)/beta1

In [27]:
years = data[:,0]
R = data[:,1]
S = data[:,2]

r = 1/R
s = 1/S
r = r.reshape(-1,1)
s = s.reshape(-1,1)

model = LinearRegression(fit_intercept=True).fit(X=s, y=r)
beta2_orig = model.coef_.item()
beta1_orig = model.intercept_.item()

point_est_theta_hat = theta_hat(beta1_orig, beta2_orig)
print(point_est_theta_hat)

150.0976343009786


In [ ]:
rng = np.random.default_rng(seed=42)

n_bootstraps = 1000
n_indv_samples = len(r)
s_r_pairs = np.concatenate((s,r), axis=1)

# Preparation for the residual bootstrap, we first calculate the residuals on the original sample
model = LinearRegression(fit_intercept=True).fit(X=s, y=r)
prediction = model.predict(s)
residuals_orig = r - prediction

bootstraps_res_betas = np.zeros([n_bootstraps, 2])
bootstraps_pairs_betas = np.zeros([n_bootstraps, 2])

n_nested_bootstraps = 100

for i in range(n_bootstraps):
    # Residual bootstrap
    bootstrap_residuals = rng.choice(residuals_orig, n_indv_samples, replace=True)
    bootstrap_res_y = prediction + bootstrap_residuals    
    model_bootstrap_res = LinearRegression(fit_intercept=True).fit(X=s, y=bootstrap_res_y)
    res_beta1 = model_bootstrap_res.intercept_.item()
    res_beta2 = model_bootstrap_res.coef_.item()
    bootstraps_res_betas[i,0] = res_beta1
    bootstraps_res_betas[i,1] = res_beta2

    res_estimate = theta_hat(res_beta1, res_beta2)

    # Pair bootstrap
    bootstrap_pairs = rng.choice(s_r_pairs, n_indv_samples, replace=True)
    model_bootstrap_pair = LinearRegression(fit_intercept=True).fit(X=bootstrap_pairs[:,0].reshape(-1,1), y=bootstrap_pairs[:,1].reshape(-1,1))
    bootstraps_pairs_betas[i,0] = model_bootstrap_pair.intercept_.item()
    bootstraps_pairs_betas[i,1] = model_bootstrap_pair.coef_.item()

    # The nested bootstrap
    for j in range(n_nested_bootstraps):
        nested_res = rng.choice(bootstrap_residuals, n_indv_samples, replace=True)
        nested_res_y = prediction + nested_res
        model_nested_res = LinearRegression(fit_intercept=True).fit(X=s, y=nested_res_y)
        nested_res_beta1 = model_bootstrap_res.intercept_.item()
        nested_res_beta2 = model_bootstrap_res.coef_.item()
        nested_res_estimate = theta_hat(nested_res_beta1, nested_res_beta2)

        rng.choice(bootstrap_pairs, n_indv_samples, replace=True)

        if i % 100 == 0 and j % 10 == 0:
            print(i,j)

In [24]:
theta_hat(bootstraps_res_betas[:,0], bootstraps_res_betas[:,1]).mean()

np.float64(150.27459494721197)